 # Test LLm interpretation of DB queries
 
Steps:
1. Interpret user questions to form valid SQL queries.
2. Execute these queries against your database.
3. Take the SQL query results and format them into a structure that can be easily used to generate a natural language response.

Limitations:
- Language
- Intent identification
- Memory
- Performance
- Security (SQL-injection, reliable results)

### TODO:
1. Train / fine tune spacy to identify intents (e.g. Tablename, dates, additional info (special words))
2. Use spacy to identify intent & generate SQL query
3. Use LLM for meaningful data output

In [1]:
# Test DB creation
import sqlite3

def setup_database():
    # Connect to SQLite database (or create it if it doesn't exist)
    conn = sqlite3.connect('test.db')
    c = conn.cursor()

    # Create the Assignments table
    c.execute('''
        CREATE TABLE IF NOT EXISTS Assignments (
            Id INTEGER PRIMARY KEY,
            targetstarttime TEXT,
            targetendtime TEXT,
            clientname TEXT,
            actualstarttime TEXT,
            actualendtime TEXT,
            date TEXT,
            notestocheck TEXT,
            employeename TEXT,
            notes TEXT
        )
    ''')

    # Sample data to insert into the table
    assignments = [
        (1, '08:00', '10:00', 'John Doe', '08:05', '09:55', '2024-04-29', 'Wefare check and shopping', 'Alice Smith', 'Completed successfully'),
        (2, '11:00', '12:00', 'Jane Smith', '11:10', '11:50', '2024-04-30', 'Confirm appointment details', 'Bob Johnson', 'Urgent follow-up needed'),
        (3, '14:00', '16:00', 'Michael Brown', '14:15', '16:05', '2024-04-30', 'Help showering and shopping', 'Clara Oswald', 'Went overtime due to unexpected issues'),
        (4, '08:00', '09:00', 'John Doe', '08:05', '09:55', '2024-04-30', '', 'Bob Johnson', 'Client did not cooperate'),
        (5, '09:00', '10:00', 'John Doe', '08:05', '09:55', '2024-04-30', 'Check the project file', 'Alice Smith', 'Completed successfully'),
        (6, '12:00', '13:00', 'Test', '08:05', '09:55', '2024-04-30', 'Checked wund', 'Alice Smith', 'Completed successfully')   
    ]

    # Insert data into the table
    c.executemany('INSERT INTO Assignments VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)', assignments)
    conn.commit()  # Commit the changes
    conn.close()  # Close the connection

    print("Database setup complete with sample data inserted.")

# Execute the function to set up the database
setup_database()


Database setup complete with sample data inserted.


### Simple keyword extraction to generate SQL Queries

In [17]:
# Approach 1
def keyword_to_sql(question):
    # Lowercase to simplify matching
    question = question.lower()

    # Basic keyword parsing for demonstration
    if 'assignments' in question and 'due for' in question:
        # Extract client name (assuming it follows 'due for')
        words = question.split()
        due_for_index = words.index('due') + 2  # gets the word after 'due for'
        client_name = words[due_for_index] + ' ' + words[due_for_index+1]

        # Example SQL query construction
        sql_query = f"SELECT * FROM Assignments WHERE clientname = '{client_name}'"
        return sql_query
    return "Query not recognized."

# Example usage
print(keyword_to_sql("What assignments are due for John Doe next week?"))

SELECT * FROM Assignments WHERE clientname = 'john doe'


### Spacy
Can recognize named entities, such as persons, companies, dates,.. -> can be refined and trained

In [34]:
# Approach 2
import spacy

# Load the English NLP model
nlp = spacy.load("de_core_news_sm")

def parse_question(question):
    doc = nlp(question)
    client_name = None
    # Identify named entities and other components
    print(doc.ents)
    for ent in doc.ents:
        if ent.label_ == "PER":
            client_name = ent.text

    if client_name:
        return f"SELECT * FROM Assignments WHERE clientname = '{client_name}'"
    return "Query not recognized."

# Example usage
#print(parse_question("Show me the assignments for John Doe next week"))
# Works well with the english model: "en_core_web_sm"

print(parse_question("Zeig mir alle Termine für John Doe in der nächsten Woche"))

(Zeig mir alle Termine, John Doe)
SELECT * FROM Assignments WHERE clientname = 'John Doe'


In [35]:
import sqlite3
import openai

def execute_sql(sql_query):
    # Connect to an SQLite database
    conn = sqlite3.connect('test.db')
    c = conn.cursor()
    c.execute(sql_query)
    results = c.fetchall()
    conn.close()
    return results

def format_response(results):
    # Use LLM to format the SQL results into a human-readable response
    prompt = "Convert these database results into a detailed, friendly explanation: " + str(results)
    response = openai.Completion.create(
        engine="text-davinci-002",
        prompt=prompt,
        max_tokens=150
    )
    answer = response.choices[0].text.strip()
    return answer

In [48]:
import speech_recognition as sr

def convert_voice_to_text():
    recognizer = sr.Recognizer()
    
    with sr.Microphone() as source:
        print("Listening...")
        audio = recognizer.listen(source)
    
        try:
            text = recognizer.recognize_google(audio, language="de-DE")#.recognize_sphinx(audio)
            print("You said: " + text)
        except sr.UnknownValueError:
            text = ""
            print("Sorry, I didn't understand that.")
        except sr.RequestError as e:
            text = ""
            print("Error; {0}".format(e))
    return text


In [53]:
from transformers import pipeline

def construct_answer(question, query_result):  
    # Load a pre-trained model and tokenizer
    generator = pipeline('text-generation', model='dbmdz/german-gpt2')

    # Example prompt
    prompt = "Frage: {0} Daten: {1}".format(question, query_result)

    # Generate response
    response = generator(prompt, max_length=250, num_return_sequences=1)
    return response
    #print(response[0]['generated_text'])

In [57]:
# Main
# Example Usage
userinput = convert_voice_to_text()
query = parse_question(userinput)

#user_question = "Employee: Alice Smith. Which clients do I need to visit tomorrow?"
#sql_query = generate_sql(query)
query_results = execute_sql(query)
#print('Query: {0}'.format(query))
response = construct_answer(userinput, query_results)
print(response[0]['generated_text'])

Listening...
You said: was sind die nächsten Termine für John Doe
(John Doe,)


Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Frage: was sind die nächsten Termine für John Doe Daten: [(1, '08:00', '10:00', 'John Doe', '08:05', '09:55', '2024-04-29', 'Wefare check and shopping', 'Alice Smith', 'Completed successfully'), (4, '08:00', '09:00', 'John Doe', '08:05', '09:55', '2024-04-30', '', 'Bob Johnson', 'Client did not cooperate'), (5, '09:00', '10:00', 'John Doe', '08:05', '09:55', '2024-04-30', 'Check the project file', 'Alice Smith', 'Completed successfully')]: [(1, '2008', '07', '08:00', '09:15', '01:25', '02:50', '03:20', '06, '07', '17, 20, 20, 20']: Wo sind Sie geboren: [(1, '08:00', '


In [ ]:
!pip install -U pip setuptools wheel
!pip install -U spacy
!python -m spacy download de_core_news_sm